# Practical 3: Classification Analysis
This notebook contains both parts:
- **Part A:** Support Vector Machine (SVM) to classify images of handwritten digits (0 to 9)
- **Part B:** K-Nearest Neighbours (KNN) on the Social Network Ads dataset, computing confusion matrix, accuracy, error rate, precision and recall

**Dataset for Part B:** https://www.kaggle.com/datasets/rakeshrau/social-network-ads (file: `Social_Network_Ads.csv`)

## Common Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_digits
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score, ConfusionMatrixDisplay)

import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

---
# PART A: SVM for Handwritten Digit Classification
**Problem Statement:** Classify images of handwritten digits into their numerical classes (0 to 9) using a Support Vector Machine. The built-in scikit-learn digits dataset is used (1797 images, each 8x8 pixels), so no download is needed.

## A1. Load the Dataset

In [ ]:
digits = load_digits()

print("Images shape :", digits.images.shape)
print("Data shape   :", digits.data.shape, "(each 8x8 image flattened into 64 features)")
print("Classes      :", np.unique(digits.target))

## A2. Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
for ax, img, label in zip(axes.ravel(), digits.images[:20], digits.target[:20]):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(f"Label: {label}", fontsize=9)
    ax.axis("off")
plt.suptitle("Sample Handwritten Digits", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
pd.Series(digits.target).value_counts().sort_index().plot(kind="bar", color="steelblue", figsize=(7, 3.5))
plt.title("Number of Samples per Digit")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

## A3. Split and Scale the Data

In [ ]:
Xd = digits.data
yd = digits.target

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    Xd, yd, test_size=0.2, random_state=42, stratify=yd)

scaler_d = StandardScaler()
Xd_train_s = scaler_d.fit_transform(Xd_train)
Xd_test_s = scaler_d.transform(Xd_test)

print("Train:", Xd_train_s.shape, " Test:", Xd_test_s.shape)

## A4. Train the SVM Classifier (RBF kernel)

In [ ]:
svm = SVC(kernel="rbf", C=10, gamma="scale", random_state=42)
svm.fit(Xd_train_s, yd_train)

yd_pred = svm.predict(Xd_test_s)
print("SVM (RBF kernel) trained.")

## A5. Evaluate the SVM

In [ ]:
acc_svm = accuracy_score(yd_test, yd_pred)
print(f"Accuracy: {acc_svm:.4f}  ({acc_svm*100:.2f}%)\n")
print(classification_report(yd_test, yd_pred))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(yd_test, yd_pred, cmap="Blues", ax=ax)
ax.set_title("SVM Confusion Matrix (Digits)")
plt.show()

## A6. Compare Different Kernels

In [ ]:
kernel_results = {}
for k in ["linear", "poly", "rbf", "sigmoid"]:
    m = SVC(kernel=k, random_state=42)
    m.fit(Xd_train_s, yd_train)
    kernel_results[k] = accuracy_score(yd_test, m.predict(Xd_test_s))

kernel_df = pd.DataFrame(kernel_results, index=["Accuracy"]).T.round(4)
display(kernel_df)

kernel_df.plot(kind="bar", legend=False, color="darkorange", figsize=(6, 3.5))
plt.title("SVM Accuracy by Kernel")
plt.ylabel("Accuracy")
plt.ylim(0.8, 1.0)
plt.xticks(rotation=0)
plt.show()

## A7. Show Some Predictions

In [ ]:
idx = np.random.RandomState(1).choice(len(Xd_test), 15, replace=False)
fig, axes = plt.subplots(3, 5, figsize=(10, 6.5))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(Xd_test[i].reshape(8, 8), cmap="gray_r")
    ok = yd_pred[i] == yd_test[i]
    ax.set_title(f"True: {yd_test[i]}  Pred: {yd_pred[i]}", color="green" if ok else "red", fontsize=10)
    ax.axis("off")
plt.suptitle("SVM Predictions (green = correct, red = wrong)")
plt.tight_layout()
plt.show()

### Part A Conclusion
The SVM with an RBF kernel classifies handwritten digits with very high accuracy (around 98%). Scaling the pixel values helps the kernel work well. The confusion matrix shows only a handful of misclassifications, and comparing kernels shows RBF, linear and polynomial all perform well while sigmoid is the weakest.

---
# PART B: KNN on Social Network Ads
**Problem Statement:** Implement the K-Nearest Neighbours algorithm on the Social Network Ads dataset to predict whether a user purchases a product after seeing an ad. Compute the confusion matrix, accuracy, error rate, precision and recall.

## B1. Load the Dataset
Download `Social_Network_Ads.csv` from Kaggle and keep it in the same folder as this notebook.

In [ ]:
ads = pd.read_csv("Social_Network_Ads.csv")
ads.head()

In [ ]:
print("Shape:", ads.shape)
ads.info()
print("\nMissing values:\n", ads.isnull().sum())

In [ ]:
ads.describe()

## B2. Explore the Data

In [ ]:
print(ads["Purchased"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="Purchased", data=ads, ax=axes[0], palette="Set2")
axes[0].set_title("Purchased (0 = No, 1 = Yes)")

sns.scatterplot(data=ads, x="Age", y="EstimatedSalary", hue="Purchased", palette="Set1", ax=axes[1])
axes[1].set_title("Age vs Estimated Salary")
plt.tight_layout()
plt.show()

## B3. Select Features and Target
Age and EstimatedSalary are the features; `User ID` is just an identifier, and Gender has little effect.

In [ ]:
X = ads[["Age", "EstimatedSalary"]].values
y = ads["Purchased"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

print("Train:", X_train.shape, " Test:", X_test.shape)

## B4. Feature Scaling
KNN uses distances, and EstimatedSalary is far larger than Age, so scaling is essential.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## B5. Choose the Best K

In [ ]:
k_values = range(1, 26)
test_acc = []
for k in k_values:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_s, y_train)
    test_acc.append(accuracy_score(y_test, m.predict(X_test_s)))

best_k = k_values[int(np.argmax(test_acc))]

plt.figure(figsize=(8, 4))
plt.plot(k_values, test_acc, marker="o")
plt.axvline(best_k, color="red", linestyle="--", label=f"Best K = {best_k}")
plt.title("Accuracy vs K")
plt.xlabel("K (number of neighbours)")
plt.ylabel("Test Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Best K:", best_k)

## B6. Train the KNN Model

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k, metric="minkowski", p=2)
knn.fit(X_train_s, y_train)
y_pred = knn.predict(X_test_s)

## B7. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Purchased", "Purchased"],
            yticklabels=["Not Purchased", "Purchased"])
plt.title("KNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(f"True Negatives (TN) : {TN}")
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")
print(f"True Positives (TP) : {TP}")

## B8. Accuracy, Error Rate, Precision and Recall

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy   : {accuracy:.4f}")
print(f"Error Rate : {error_rate:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"F1 Score   : {f1:.4f}")

### Manual verification using the confusion matrix formulas

In [ ]:
print("Accuracy   = (TP + TN) / Total   =", round((TP + TN) / (TP + TN + FP + FN), 4))
print("Error Rate = (FP + FN) / Total   =", round((FP + FN) / (TP + TN + FP + FN), 4))
print("Precision  = TP / (TP + FP)      =", round(TP / (TP + FP), 4))
print("Recall     = TP / (TP + FN)      =", round(TP / (TP + FN), 4))

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Not Purchased", "Purchased"]))

## B9. Decision Boundary Visualization

In [ ]:
def plot_boundary(X_set, y_set, title):
    x1, x2 = np.meshgrid(
        np.arange(X_set[:, 0].min() - 1, X_set[:, 0].max() + 1, 0.02),
        np.arange(X_set[:, 1].min() - 1, X_set[:, 1].max() + 1, 0.02))
    Z = knn.predict(np.c_[x1.ravel(), x2.ravel()]).reshape(x1.shape)
    plt.figure(figsize=(7, 5))
    plt.contourf(x1, x2, Z, alpha=0.3, cmap="RdYlGn")
    for cls, color in zip([0, 1], ["red", "green"]):
        plt.scatter(X_set[y_set == cls, 0], X_set[y_set == cls, 1],
                    c=color, edgecolor="k", s=25, label=f"Class {cls}")
    plt.title(title)
    plt.xlabel("Age (scaled)")
    plt.ylabel("Estimated Salary (scaled)")
    plt.legend()
    plt.show()

plot_boundary(X_test_s, y_test, f"KNN Decision Boundary (K = {best_k}) - Test Set")

### Part B Conclusion
KNN with scaled Age and EstimatedSalary classifies purchases with high accuracy (typically around 90% or more). The confusion matrix gives TP, TN, FP and FN, from which accuracy, error rate (1 - accuracy), precision (how many predicted buyers really bought) and recall (how many actual buyers were found) are calculated. Feature scaling was essential because KNN depends on distance.